This notebook is for modeling and evaluating technique classification from the SemEval dataset.

In [5]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset
import evaluate
from sklearn.model_selection import GroupShuffleSplit

In [2]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_bert_scanner"

In [4]:
#Load technique classification data
df_tc = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df_tc.head()

,article_id,text_content,span_text,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,0.250000,0,1.000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,0.483333,0,0.863636,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,0.000000,0,1.000000,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
#Split the data by article rather than by span to avoid any data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_tc, groups=df_tc['article_id']))
train_df = df_tc.iloc[train_idx].reset_index(drop=True)
test_df = df_tc.iloc[test_idx].reset_index(drop=True)

#Verify the split
print(f"Total spans: {len(df_tc)}, total articles: {df_tc['article_id'].nunique()}")
print(f"Train spans: {len(train_df)} ({train_df['article_id'].nunique()} articles)")
print(f"Test spans:  {len(test_df)} ({test_df['article_id'].nunique()} articles)")

#Check for leakage (should be 0)
overlap = set(train_df['article_id']).intersection(set(test_df['article_id']))
print(f"Number of overlapping articles: {len(overlap)}")

Total spans: 7587, total articles: 357
Train spans: 5856 (285 articles)
Test spans:  1731 (72 articles)
Number of overlapping articles: 0
